# 02 · Simulated account server

Run top-to-bottom in a fresh project Python kernel. Cells tagged `live` contact external services and may incur model costs. See [source attribution](../ATTRIBUTION.md).

In [1]:
from pathlib import Path
import os, sys, tempfile
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "backend/accounts_server.py").exists())
sys.path.insert(0, str(ROOT))
# Fresh temporary data for this notebook; never reset the dashboard's accounts.
notebook_data = tempfile.TemporaryDirectory()
os.environ["MCP_DATA_DIR"] = notebook_data.name
os.environ["MARKET_MODE"] = "fixture"
os.environ["ENABLE_PUSH"] = "false"
print("Using isolated fixture data.")

Using isolated fixture data.


## Record a paper position

In [2]:
from backend.accounts import Account
account = Account.get("Notebook")
account.buy_shares("AAPL", 5, "Demonstrate a paper position")
assert account.holdings == {"AAPL": 5}
assert abs(account.balance - 9499) < 1e-6
account.list_transactions()

[{'symbol': 'AAPL',
  'quantity': 5,
  'price': 100.2,
  'timestamp': '2026-10-08 09:26:40',
  'rationale': 'Demonstrate a paper position'}]

## Reject invalid orders without modifying the account

In [3]:
before = account.model_dump()
try:
    account.sell_shares("AAPL", -1, "Invalid order")
except ValueError as error:
    print(error)
else:
    raise AssertionError("Invalid order was accepted")
assert account.model_dump() == before

Quantity must be a positive whole number


## Separate cash deposits from profit

In [4]:
before_pnl = account.get_profit_loss()
account.deposit(500)
assert abs(account.get_profit_loss() - before_pnl) < 1e-6
account.sell_shares("AAPL", 5, "Close the demo position")
assert abs(account.get_profit_loss() + 2) < 1e-6
account.report()

'{"name": "notebook", "balance": 10498.0, "net_deposits": 500.0, "strategy": "", "holdings": {}, "transactions": [{"symbol": "AAPL", "quantity": 5, "price": 100.2, "timestamp": "2026-10-08 09:26:40", "rationale": "Demonstrate a paper position"}, {"symbol": "AAPL", "quantity": -5, "price": 99.8, "timestamp": "2026-10-08 09:26:40", "rationale": "Close the demo position"}], "portfolio_value_time_series": [["2026-10-08 09:26:40", 9999.0], ["2026-10-08 09:26:40", 10498.0], ["2026-10-08 09:26:40", 10498.0]], "total_portfolio_value": 10498.0, "total_profit_loss": -2.0}'